# Part 1. Equation of a Slime

How many late days are you using for this assignment?

0

In [2]:
# Imports section
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

## 1. Loading the dataset

In [3]:
# Using pandas load the dataset
df = pd.read_csv('science_data_large.csv')

# Output the first 15 rows of the data
print("First 15 rows of the dataset:")
print(df.head(15))

# Display a summary of the table information (number of datapoints, etc.)
print("\nDataset Summary:")
print(df.info())

First 15 rows of the dataset:
    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04

Dataset Summary:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Temperature °C  1000 non-null   int64  
 1   Mols KCL        100

## 2. Splitting the dataset

In [5]:
# Take the pandas dataset and split it into our features (X) and label (y)
X = df[['Temperature °C', 'Mols KCL']]  # Features
y = df['Size nm^3']  # Label

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
# For grading consistency use random_state=42 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

## 3. Perform a Linear Regression

In [7]:
# Use sklearn to train a model on the training set
model = LinearRegression()
model.fit(X_train, y_train)

sample_point = pd.DataFrame([[500, 500]], columns=['Temperature °C', 'Mols KCL'])
prediction = model.predict(sample_point)
print(f"Predicted size for Temperature=500°C, KCL=500 mols: {prediction[0]:.2f} nm^3")

# Get the score (R²) for training and test sets
train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)
print(f"\nTraining R² Score: {train_score:.4f}")
print(f"Test R² Score: {test_score:.4f}")

# Extract the coefficients and intercept from the model
coef_temp = model.coef_[0]
coef_kcl = model.coef_[1]
intercept = model.intercept_

print(f"\nEquation coefficients:")
print(f"Temperature coefficient: {coef_temp:.2f}")
print(f"KCL coefficient: {coef_kcl:.2f}") 
print(f"Intercept: {intercept:.2f}")

Predicted size for Temperature=500°C, KCL=500 mols: 540029.26 nm^3

Training R² Score: 0.8610
Test R² Score: 0.8552

Equation coefficients:
Temperature coefficient: 866.15
KCL coefficient: 1032.70
Intercept: -409391.48


Write the linear equation of a slime: (example equation: $E = mc^2$)

$h(x) = 866.15T + 1032.70K - 409391.48$

where:
- T = Temperature in °C
- K = Mols of KCL

Report on score and explain meaning:

The model achieved an R² score of 0.8610 on the training data and 0.8552 on the test data. These scores indicate:

1. The model explains about 86% of the variance in slime size, showing a strong but not perfect fit
2. The similar scores between training (0.8610) and test (0.8552) suggest the model generalizes well without overfitting
3. The remaining ~14% of variance indicates there may be:
   - Other factors affecting slime size not captured by temperature and KCL
   - Non-linear relationships that the linear model cannot capture
   - Natural random variation in slime growth

Overall, while not perfect, this is a reasonably good model for predicting slime size based on temperature and KCL concentration.

## 4. Use Cross Validation

In [9]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
# For grading consistency use n_splits=5 and random_state=42

from sklearn.model_selection import KFold

# Create KFold cross-validator with random_state
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=kf)

# Calculate mean and standard deviation of the scores
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

print("Cross Validation Scores:", cv_scores)
print(f"Mean R² Score: {cv_mean:.4f} (+/- {cv_std*2:.4f})")

Cross Validation Scores: [0.86151889 0.82742341 0.87195173 0.88166206 0.85609101]
Mean R² Score: 0.8597 (+/- 0.0368)


Write findings here:

The cross-validation results show:
- Individual scores across 5 folds: [0.8615, 0.8274, 0.8720, 0.8817, 0.8561]
- Mean R² Score: 0.8597
- Score variation: ±0.0368 (two standard deviations)

These results indicate:

1. **Consistent Performance**: The model maintains R² scores between 0.83-0.88 across different data splits, showing stable predictive power.

2. **Reliability**: The relatively small standard deviation (±0.0368) suggests the model's performance is consistent and not heavily dependent on how the data is split.

3. **Validation of Earlier Results**: The mean cross-validation score (0.8597) aligns well with our earlier training (0.8610) and test (0.8552) scores, which confirms the model's predictive ability.

4. **Generalization**: The consistent scores across different data splits suggest the model will likely perform similarly on new, unseen data within the same distribution.

This cross-validation reinforces our earlier conclusion that the linear model is a reliable predictor of slime size, explaining about 86% of the variance in the data.

## 5. Using Polynomial Regression

In [10]:
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)

# Create and train the polynomial regression model
poly_model = LinearRegression()
poly_model.fit(poly.fit_transform(X_train), y_train)

poly_train_score = poly_model.score(poly.transform(X_train), y_train)
poly_test_score = poly_model.score(poly.transform(X_test), y_test)

# Perform cross-validation
cv_scores_poly = cross_val_score(poly_model, X_poly, y, cv=kf)
cv_mean_poly = cv_scores_poly.mean()
cv_std_poly = cv_scores_poly.std()

# Print results
print("Polynomial Regression Results:")
print(f"Training R² Score: {poly_train_score:.4f}")
print(f"Test R² Score: {poly_test_score:.4f}")
print(f"Cross Validation Scores: {cv_scores_poly}")
print(f"Mean CV R² Score: {cv_mean_poly:.4f} (+/- {cv_std_poly*2:.4f})")

feature_names = ['1', 'T', 'K', 'T²', 'T*K', 'K²']
coefficients = dict(zip(feature_names, poly_model.coef_))

print("\nEquation coefficients:")
for name, coef in coefficients.items():
    print(f"{name}: {coef:.2f}")
print(f"Intercept: {poly_model.intercept_:.2f}")

Polynomial Regression Results:
Training R² Score: 1.0000
Test R² Score: 1.0000
Cross Validation Scores: [1. 1. 1. 1. 1.]
Mean CV R² Score: 1.0000 (+/- 0.0000)

Equation coefficients:
1: 0.00
T: 12.00
K: -0.00
T²: 0.00
T*K: 2.00
K²: 0.03
Intercept: 0.00


Write the polynomial equation of a slime: (example equation: $E = mc^2$)

$h(x) = 12T + 2TK + 0.03K^2$

where:
- T = Temperature in °C
- K = Mols of KCL
- TK represents the interaction term between Temperature and KCL

Report on the score and interpret:

The polynomial regression model achieved:
- Training R² Score: 1.0000
- Test R² Score: 1.0000
- Cross Validation Mean Score: 1.0000 (±0.0000)

These perfect scores (1.0000) indicate:

1. **Perfect Fit**: The model explains 100% of the variance in slime size, suggesting the true relationship between variables is polynomial
2. **No Overfitting**: The identical scores across training, testing, and cross-validation suggest we've found the true underlying pattern
3. **Verification**: The zero standard deviation in cross-validation confirms the relationship is consistent across all data splits

This is a significant improvement over the linear model (R² ≈ 0.86), which means that the slime growth follows a precise quadratic pattern.

# Part 2. Chronic Kidney Disease Prediction via Classification

Create code and markdown cells as needed to perform classification and report on your results

In [ ]:
# Load the dataset. Then train and evaluate the classification models.
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('ckd_feature_subset.csv')
X = df.drop('Target_ckd', axis=1)
y = df['Target_ckd']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Initialize models
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'SVM': SVC(random_state=42),
    'KNN': KNeighborsClassifier(),
    'Neural Network': MLPClassifier(hidden_layer_sizes=(10,), random_state=42)
}

# Perform cross-validation for each model
results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y, cv=kf, scoring='accuracy')
    results[name] = {
        'Mean Accuracy': scores.mean(),
        'Std Dev': scores.std()
    }

# Display results
results_df = pd.DataFrame(results).T
print("\nModel Performance Results:")
print(results_df.round(4))

/Users/<redacted>/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/<redacted>/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/<redacted>/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



Model Performance Results:
                     Mean Accuracy  Std Dev
Logistic Regression         0.9480   0.0436
SVM                         0.9671   0.0293
KNN                         0.9215   0.0604
Neural Network              0.8830   0.0802


/Users/<redacted>/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/<redacted>/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


## Results and Conclusion for Classification Experiments

## Model Performance Comparison

| Model | Mean Accuracy | Standard Deviation |
|-------|--------------|-------------------|
| SVM | 0.9671 | ±0.0293 |
| Logistic Regression | 0.9480 | ±0.0436 |
| KNN | 0.9215 | ±0.0604 |
| Neural Network | 0.8830 | ±0.0802 |

## Analysis

1. **Best Performing Model**: Support Vector Machine (SVM)
   - Highest accuracy of 96.71%
   - Most stable performance with lowest standard deviation (±0.0293)
   - Likely performs best due to:
     - Effective handling of non-linear relationships in the data
     - Strong generalization capabilities
     - Robust performance with standardized features

2. **Model Comparison**:
   - SVM and Logistic Regression showed superior performance (>94% accuracy)
   - KNN performed reasonably well (92.15%) but with higher variance
   - Basic Neural Network showed lowest performance, suggesting need for optimization

3. **Conclusion**:
   - SVM would be most appropriate for real-world deployment due to:
     - Highest accuracy and stability
     - Good balance of performance vs. complexity

In [4]:
# Experiments with Neural Network.
# Neural Network Parameter Experiments
nn_params = [
    {'hidden_layer_sizes': (5,), 'name': 'Small Network'},
    {'hidden_layer_sizes': (20,), 'name': 'Medium Network'},
    {'hidden_layer_sizes': (10,10), 'name': 'Deep Network'}
]

nn_results = {}
for params in nn_params:
    name = params.pop('name')
    model = MLPClassifier(random_state=42, max_iter=1000, **params)
    scores = cross_val_score(model, X_scaled, y, cv=kf, scoring='accuracy')
    nn_results[name] = {
        'Mean Accuracy': scores.mean(),
        'Std Dev': scores.std()
    }

# Display neural network results
nn_results_df = pd.DataFrame(nn_results).T
print("\nNeural Network Configurations Results:")
print(nn_results_df.round(4))

/Users/<redacted>/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/<redacted>/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/<redacted>/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/<redacted>/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warning


Neural Network Configurations Results:
                Mean Accuracy  Std Dev
Small Network          0.9090   0.0469
Medium Network         0.9544   0.0436
Deep Network           0.9611   0.0474


## Results and Conclusion for Neural Network Experiments

# Neural Network Configuration Analysis

## Configuration Results

| Configuration | Mean Accuracy | Standard Deviation |
|---------------|--------------|-------------------|
| Deep Network (10,10) | 0.9611 | ±0.0474 |
| Medium Network (20) | 0.9544 | ±0.0436 |
| Small Network (5) | 0.9090 | ±0.0469 |

## Analysis

1. **Best Configuration**: Deep Network (two layers of 10 neurons)
   - Achieved highest accuracy of 96.11%
   - Maintained reasonable stability (±0.0474)
   - Shows benefit of deeper architecture for this problem

2. **Parameter Impact**:
   - Network depth had more impact than width
   - Adding a second layer (Deep Network) improved performance over single wider layer (Medium Network)
   - Small network (5 neurons) showed clear signs of underfitting
   - Larger architectures improved accuracy without significant overfitting

3. **Conclusion**:
   - Use Deep Network configuration for optimal performance
   - Balance between model complexity and diminishing returns suggests Deep Network (10,10) is optimal